<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/08-io-devices-and-event-driven-execution.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **I/O Devices and Event-Driven Execution**

The previous chapter ended with a **major page fault** waiting for storage. This chapter follows that wait across the rest of the machine. A request must cross a protection boundary, find the kernel object behind a file descriptor, enter a subsystem, reach a device driver, become a controller command, move data, and eventually produce a completion that wakes a task or notifies an event loop.

I/O is difficult because the CPU and a device are independent agents. A function call appears sequential, but the physical operation has two separated moments:

1. **submission**, when software describes and publishes work; and
2. **completion**, when the device or kernel reports what actually happened.

Between those moments, the calling thread might sleep, the CPU might run another process, the controller might reorder commands, and the device might disappear. This is why I/O cannot be understood as simply "copy bytes and return." It is a problem of **concurrency, ownership, notification, flow control, and recovery**.

A useful mental model for the whole chapter is:

> **An I/O interface is a contract for submitting operations and observing their results. It does not make the external device synchronous, reliable, or semantically identical to every other object.**

The chapter develops that contract from basic registers and interrupts to readiness-based event loops and `io_uring`. File-system naming, inode structure, block allocation, journaling, and `fsync()` semantics belong to the next chapter. Here, a regular file is one possible source or destination in the I/O pipeline.

### **The Operating System's I/O Problem**

An operating system sits between programs that want a small, stable API and hardware that differs in transfer unit, latency, ordering, failure behavior, and control protocol. A keyboard may deliver a few bytes unpredictably. An NVMe drive may accept thousands of commands across multiple queues. A network interface may DMA bursts of packets into memory. A GPU may run long command streams and signal fences later. Treating these devices identically at the hardware level would be impossible; exposing every hardware protocol directly to every application would be unsafe and unmaintainable.

The kernel therefore solves several related mismatches:

| Mismatch | What the kernel must provide | Failure if it does not |
|---|---|---|
| **Interface** | Stable operations such as open, read, write, control, map, and wait | Every program must know each controller protocol |
| **Timing** | Sleep, notification, and asynchronous progress while devices are slow | CPU time is wasted spinning or the whole program freezes |
| **Granularity** | Buffering and translation between bytes, blocks, packets, and device descriptors | Tiny requests dominate overhead or boundaries are corrupted |
| **Protection** | Validate requests and restrict DMA and register access | A process or device can overwrite unrelated kernel memory |
| **Concurrency** | Queueing, ordering, cancellation, and ownership rules | Results are associated with the wrong request or buffers are reused early |
| **Failure** | Timeouts, error propagation, reset, and hotplug handling | A lost interrupt or removed device leaves work stuck forever |
| **Policy** | Fairness, priorities, queue limits, and accounting | One workload monopolizes the device or consumes unbounded memory |

I/O performance also spans several time scales. A cached read may complete in microseconds without reaching hardware. A storage request may wait behind other work before the device services it. A human-facing device may remain idle for seconds. An API that is efficient for one regime can be wasteful in another, so the kernel offers both blocking and event-driven forms rather than one universal waiting mechanism.

#### **Uniform Interfaces Over Heterogeneous Devices**

Unix-like systems make a **file descriptor** a small process-local handle to a kernel object. The descriptor is not the object itself and is not necessarily a disk file. It selects an entry in the process's descriptor table, which refers to an open kernel object carrying access mode, current state, operation methods, and references to the underlying resource.

This design is powerful because the same operations can be composed across files, pipes, terminals, sockets, and many devices. Redirection works because a program can write to descriptor 1 without knowing whether it currently denotes a terminal, pipe, regular file, or socket. Readiness APIs can wait on several kinds of descriptors together. Access control is checked when the handle is created and again when an operation requires it.

![Uniform calls cross an abstraction boundary and then recover object-specific behavior.](assets/io-uniform-interface-boundaries.svg){fig-alt="Applications call read, write, ioctl, mmap, polling, or io_uring through file descriptors; kernel objects and drivers translate those calls to regular files, pipes, sockets, and devices with different semantics." width="96%"}

*Figure: original explanatory diagram informed by the [Linux VFS documentation](https://docs.kernel.org/filesystems/vfs.html) and the [OSTEP I/O devices chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/file-devices.pdf).*

Uniform syntax does **not** imply uniform semantics:

| Object | What `read()` consumes | Meaning of `0` | Is seeking meaningful? | Typical readiness caveat |
|---|---|---|---|---|
| Regular file | Bytes at a file offset | End of file | Usually yes | Usually appears immediately ready even if later storage I/O may block |
| Pipe/FIFO | Bytes produced by writers | All writers closed after buffered bytes drain | No | Readable can mean data **or** EOF |
| TCP socket | Ordered byte stream from a peer | Peer performed an orderly shutdown | No | Readable can mean data, shutdown, or pending error |
| Datagram socket | One queued message at a time | A zero-length datagram can be valid data | No | Message boundaries and truncation matter |
| Terminal/device node | Driver-defined byte stream or records | Device-specific | Often no | `ioctl()` and device state can be essential |

The abstraction should therefore be read as "a common vocabulary with object-specific contracts." Correct software still asks whether operations can be partial, what establishes end-of-stream, whether order is preserved, and which state changes are durable. `ioctl()` exists precisely because not every useful control operation can be reduced to byte transfer, although each ioctl command must be designed and validated carefully.

### **Device Controllers and Drivers**

A **device** is the physical or virtual resource that performs work. A **controller** is hardware or firmware that exposes a programmable interface to the host. A **device driver** is privileged software that translates kernel requests into that controller interface and translates controller events back into kernel results.

The boundaries vary. A simple timer may expose a handful of registers. A modern SSD contains processors, firmware, caches, and many command queues. A virtual device may be implemented by a hypervisor rather than physical electronics. The useful distinction is functional: the driver runs as host software; the controller accepts commands and progresses independently.

#### **Registers, Queues, and Command Completion**

The smallest controller model has three conceptual register classes:

- a **status register** says whether the device is busy, ready, complete, or in error;
- a **command register** tells the controller what operation to start; and
- one or more **data registers** carry payload bytes or parameters.

Registers are often exposed through **memory-mapped I/O (MMIO)**. Their physical addresses are mapped into a privileged virtual range, but loads and stores to that range have device semantics rather than ordinary RAM semantics. Architectures and kernels provide special accessors because compiler optimization, CPU reordering, bus ordering, register width, and endianness all matter. A write may be **posted**, meaning the CPU instruction retires before the controller has observed the write. Reading a status register or using an architecture-specific barrier may be required at a protocol-defined point.

Modern high-throughput devices avoid programming every operation through several individual register writes. Instead, host memory holds **descriptors**. A descriptor may contain an operation code, DMA address, byte count, flags, and request identity. Software fills one or more descriptors, publishes a producer index, and writes a small MMIO **doorbell**. The controller fetches descriptors, performs work, and writes completion records into another queue.

![A driver publishes descriptors and a controller later publishes completions.](assets/controller-register-command-queues.svg){fig-alt="A driver prepares and publishes a descriptor ring entry, orders memory before ringing an MMIO doorbell, and later consumes a completion written by a concurrently executing controller." width="96%"}

*Figure: original explanatory diagram based on the register protocol in [OSTEP I/O Devices](https://pages.cs.wisc.edu/~remzi/OSTEP/file-devices.pdf), the Linux [DMA API guide](https://docs.kernel.org/core-api/dma-api-howto.html), and the queue model documented by [NVMe](https://nvmexpress.org/specifications/).*

Queue correctness depends on explicit **ownership transitions**. Before submission, the driver owns and may modify a descriptor and its buffer. After publishing the descriptor, the controller may read or write them; the driver must not recycle those resources merely because the submission function returned. Only a matching completion, successful cancellation, or completed reset protocol returns ownership.

A simplified producer protocol is:

```text
descriptor[i] <- operation, DMA address, length, request_id
memory_barrier()              // descriptor must become visible first
submission_tail <- next(i)    // publish ownership to the controller
write_mmio(DOORBELL, tail)    // tell the controller to look

... controller runs concurrently ...

wait until completion[j].request_id is valid
memory_barrier()              // observe completion fields before reuse
consume result
completion_head <- next(j)
```

This pseudocode does not prescribe one universal barrier. The correct primitive depends on the architecture, whether memory is coherent, the DMA API, and the controller specification. "Coherent DMA" means CPU and device observe a coherent memory view; it does not remove all ordering requirements between descriptor fields and an ownership bit.

Queue depth also changes performance. If an operation's average latency is $W$ seconds and the system needs throughput $\lambda$ operations per second, Little's law suggests an average of

$$
L = \lambda W
$$

operations must be in the system. For example, sustaining 100,000 operations/s with 200 microseconds of end-to-end latency requires roughly $100{,}000 \times 0.0002 = 20$ operations in flight on average. More queueing can expose device parallelism, but queueing beyond useful parallelism mostly increases waiting time.

#### **Driver Responsibilities and Trust**

A driver's work begins before the first `read()` and continues after the last one. Through the kernel's driver model, it normally:

1. matches a discovered device to a supported driver;
2. probes capabilities and validates firmware-visible state;
3. claims MMIO ranges, interrupt vectors, DMA resources, clocks, and queues;
4. registers the device with an appropriate kernel subsystem;
5. translates requests, enforces limits, and tracks every outstanding operation;
6. handles interrupts or polling, decodes completion status, and releases resources;
7. participates in suspend, resume, reset, error recovery, and hot removal; and
8. stops admission and drains callbacks before teardown.

![Driver correctness spans discovery, steady-state service, recovery, teardown, and security.](assets/driver-lifecycle-trust-boundary.svg){fig-alt="A driver discovers, probes, claims resources, serves, recovers, and removes a device while validating untrusted inputs, constraining DMA, and preventing use-after-removal." width="96%"}

*Figure: original explanatory diagram based on the Linux [driver-model overview](https://docs.kernel.org/driver-api/driver-model/overview.html), [generic IRQ documentation](https://docs.kernel.org/core-api/genericirq.html), and [DMA API guide](https://docs.kernel.org/core-api/dma-api-howto.html).*

Drivers are part of the trusted kernel on many systems. A bounds error, stale pointer, or double completion can corrupt the whole machine. They also consume data produced by hardware and firmware, which should not automatically be considered trustworthy. Defensive drivers validate descriptor lengths and indexes, confirm that completions refer to live requests, limit mapped DMA ranges, and cope with impossible-looking status values.

An **IOMMU** strengthens this boundary. Without it, a bus-mastering device may be able to DMA to broad physical-address ranges. With an IOMMU, the kernel establishes device-visible mappings so a device can reach only selected buffers. This is analogous to virtual memory for device transactions, although the address spaces, caches, and invalidation mechanisms are distinct from a process page table.

### **The Path of an I/O System Call**

Consider a call `read(fd, buffer, count)`. The exact functions differ between kernels and object types, but the logical path is stable:

1. **Enter the kernel.** The system-call mechanism changes privilege and transfers control to a kernel entry point while preserving the calling context.
2. **Validate scalar arguments.** The kernel checks the descriptor number, count, access mode, object state, and arithmetic for overflow.
3. **Resolve the descriptor.** The process descriptor table yields a referenced open object and its operation methods.
4. **Validate the userspace range.** The destination must be writable for the requested range. The kernel may later copy to it, pin/map pages for direct access, or use an intermediate buffer.
5. **Ask the relevant subsystem.** A file, pipe, socket, terminal, and device follow different internal paths. A cache or in-kernel queue may satisfy the request immediately.
6. **Build lower-level work on a miss.** The subsystem allocates request metadata, maps buffers for DMA if needed, and submits work to a driver/controller queue.
7. **Choose a waiting policy.** A blocking call can put the task on a wait queue. A nonblocking call can return `EAGAIN`. An asynchronous interface records how a later completion should be reported.
8. **Handle completion.** An interrupt, polling loop, or kernel worker recognizes the result, records bytes/error status, releases mappings, and wakes or notifies interested consumers.
9. **Return an observable result.** The kernel may copy data to userspace and returns a positive byte count, zero, or an error. The result may be shorter than `count`.

![An I/O request follows a submission path and a later completion path.](assets/io-system-call-completion-animated.svg){fig-alt="Animated request and completion tokens show read submission through validation, subsystem, driver, and controller, then completion through request finishing, notification, rescheduling, and return." width="96%"}

*Figure: original explanatory animation synthesizing the [Linux VFS](https://docs.kernel.org/filesystems/vfs.html), [generic IRQ](https://docs.kernel.org/core-api/genericirq.html), and [DMA API](https://docs.kernel.org/core-api/dma-api-howto.html) execution paths.*

The fast and slow paths must have the same external contract. A page-cache hit can copy bytes and return without a physical device. A pipe read can consume bytes already in a kernel ring. A socket read can consume data that arrived earlier. Only a miss or empty queue needs to wait for lower-level progress.

That distinction matters when measuring latency. A `read()` duration can include validation, lock contention, queueing, device service, completion handling, wakeup delay, and copying. Conversely, a short `read()` can consume data whose device transfer happened before the timer started. System-call timing alone does not identify which layer was slow.

### **Programmed I/O, Polling, Interrupts, and DMA**

Four terms are often taught together but answer two different questions:

- **Programmed I/O (PIO) versus DMA:** who moves the payload bytes?
- **Polling versus interrupts:** how does the CPU learn that state changed or work completed?

![PIO and DMA concern payload movement; polling and interrupts concern notification.](assets/pio-poll-interrupt-dma.svg){fig-alt="Four panels show the CPU moving payload via programmed I/O, repeatedly checking a status register, receiving a device interrupt, and configuring DMA so a device transfers directly to memory before completion notification." width="97%"}

*Figure: original explanatory diagram based on [OSTEP I/O Devices](https://pages.cs.wisc.edu/~remzi/OSTEP/file-devices.pdf) and the Linux [DMA API guide](https://docs.kernel.org/core-api/dma-api-howto.html).*

With **PIO**, CPU instructions explicitly move each payload word through a device register. It is simple and remains suitable for tiny control values, but a large transfer consumes many CPU instructions and occupies the interconnect inefficiently.

With **polling**, the CPU repeatedly reads a status location until a condition changes. Polling is not inherently primitive: when events arrive continuously and each status check is cheap, a dedicated polling loop can have lower latency and fewer context disruptions than an interrupt per event. Its cost is the CPU time spent checking when there is no useful work.

With an **interrupt**, the controller sends an asynchronous notification through an interrupt controller. The processor saves enough execution state, enters a registered handler, acknowledges or masks the interrupt as required, and arranges further processing. The hard-interrupt portion should be bounded because it delays other work. Kernels commonly defer larger tasks to threaded handlers, softirqs, work queues, or subsystem-specific poll loops.

With **DMA**, the CPU prepares a transfer and the device/controller moves payload between the device and memory. A DMA-capable device does not normally understand a process virtual address. The Linux DMA model distinguishes:

$$
\text{CPU virtual address } X
\longrightarrow
\text{CPU physical address } Y
\longrightarrow
\text{device DMA address } Z.
$$

The last translation may be identity-like on a simple system or may pass through an IOMMU. A driver asks the DMA API to create the mapping instead of assuming these addresses are equal. It also obeys mapping direction and lifetime so caches, bounce buffers, and IOMMU entries can be managed correctly.

The total latency of a bulk transfer can be decomposed approximately as

$$
T_{operation} = T_{setup} + T_{queue} + \frac{B}{R} + T_{completion},
$$

where $B$ is the number of bytes and $R$ is the effective transfer rate. DMA primarily reduces CPU work per byte; it does not eliminate setup, queueing, device service, or completion cost. For very small $B$, setup can dominate, which is why controllers batch commands and operating systems coalesce work.

| Mechanism | CPU work while data moves | Idle behavior | Strong use case | Main risk |
|---|---|---|---|---|
| PIO | CPU executes payload loads/stores | No independent transfer | Small control/status data | Poor bulk throughput and high CPU cost |
| DMA | CPU sets up and finishes transfer | Device works independently | Blocks, packets, audio/video buffers | Mapping, ownership, ordering, and isolation complexity |
| Polling | CPU repeatedly checks progress | Consumes CPU even with no event | Sustained high event rate, dedicated core | Power use and starvation |
| Interrupt | CPU is notified on event | Nearly no notification work when idle | Sparse or bursty events | Fixed interrupt cost and interrupt storms |

#### **Choosing Between Interrupts and Polling**

Suppose a status probe costs $c_p$ CPU cycles and occurs every $\Delta$ seconds during a wait of $T_w$. A rough polling cost is

$$
C_{poll} \approx \frac{T_w}{\Delta}c_p.
$$

If $N$ events each incur interrupt entry, acknowledgement, cache disruption, and deferred processing cost $c_i$, then

$$
C_{interrupt} \approx N c_i.
$$

These are not complete performance models, but they expose the crossover. Sparse events make $C_{poll}$ mostly wasted probes. Dense events make $C_{interrupt}$ repeat fixed overhead many times. The crossover changes with batching, CPU frequency, cache locality, interrupt affinity, latency requirements, and whether a core can be dedicated.

![Sparse events favor interrupts, sustained load can favor polling, and hybrid paths switch between them.](assets/interrupt-polling-hybrid.svg){fig-alt="A conceptual cost crossover compares polling and interrupts, while a Linux NAPI-style timeline starts with an interrupt, masks further interrupts, polls to a budget, and unmasks when the queue drains." width="96%"}

*Figure: original explanatory diagram based on the Linux [NAPI documentation](https://docs.kernel.org/networking/napi.html) and [generic IRQ documentation](https://docs.kernel.org/core-api/genericirq.html). The cost curves are conceptual, not benchmark data.*

Production systems frequently use a **hybrid**:

- an interrupt wakes the CPU when an idle queue first receives work;
- the handler masks or suppresses further notifications;
- software polls and drains several completions up to a budget;
- if work remains, polling is rescheduled for fairness;
- when the queue drains, interrupts are enabled again.

Linux NAPI applies this pattern to network receive processing. Storage drivers also batch completions and modern controllers provide multiple interrupt vectors so queues can be assigned to CPUs. **Interrupt coalescing** reduces interrupt rate by waiting for several events or a short timer, trading lower CPU overhead for additional notification latency. There is no context-free best setting: an interactive RPC service and a bulk-transfer server optimize different tails of the latency distribution.

### **Blocking and Nonblocking I/O**

A **blocking operation** allows the kernel to suspend the calling thread when the operation cannot currently make progress. It does not freeze the CPU and it does not necessarily freeze the whole process: the scheduler runs another runnable thread or process. Blocking is often the clearest model when one thread owns one sequential flow.

A **nonblocking operation** returns control instead of sleeping when progress would require waiting. On Unix-like systems, `O_NONBLOCK` commonly causes an operation to return `-1` with `errno == EAGAIN` or `EWOULDBLOCK`. This is not a failure of the connection or device; it means "the requested operation cannot make progress now under this waiting policy."

![Blocking sleeps and rechecks; nonblocking returns EAGAIN, while both share data, EOF, partial, and error outcomes.](assets/blocking-nonblocking-read-state.svg){fig-alt="A read decision tree checks whether progress is possible; a blocking descriptor sleeps and rechecks, a nonblocking descriptor returns EAGAIN, and ready operations return bytes, EOF, or an error." width="96%"}

*Figure: original explanatory diagram based on Linux [`open(2)`](https://man7.org/linux/man-pages/man2/open.2.html), [`read(2)`](https://man7.org/linux/man-pages/man2/read.2.html), and [`poll(2)`](https://man7.org/linux/man-pages/man2/poll.2.html) semantics.*

For a byte-stream `read()`, callers must distinguish all of the following:

| Result | Meaning | Normal response |
|---|---|---|
| `n > 0` | `n` bytes were transferred; `n` may be less than requested | Process those bytes and continue according to the protocol |
| `n == 0` | End of file / orderly stream shutdown for the relevant object | Finish input; do not spin retrying the same EOF |
| `-1, EAGAIN` | No progress now under nonblocking policy | Return to event loop or apply backoff |
| `-1, EINTR` | A signal interrupted the call before a result was produced | Retry only if application cancellation/deadline policy allows it |
| `-1, other errno` | Persistent or object-specific error | Propagate, recover, or close according to the contract |

Partial success is fundamental. A successful `read()` or `write()` promises only the returned number of bytes, not the requested number. A socket send buffer may accept a prefix. A pipe may have limited free space. A signal can arrive after some bytes have transferred. Correct code advances by the returned count and keeps the remaining state explicitly.

Nonblocking mode is most useful for objects whose availability genuinely changes, such as sockets, pipes, terminals, and event descriptors. Linux documents that `O_NONBLOCK` currently has no useful effect on ordinary file and block-device I/O: even if a descriptor is marked nonblocking, a regular-file operation may still block on page faults, storage, allocation, or metadata. Event-driven applications often use worker threads or a true asynchronous interface for such work.

Finally, readiness is a **hint about state at one instant**, not a reservation. Another thread can consume the data before the current thread calls `read()`, or an error can replace the expected condition. Event-driven code therefore keeps descriptors nonblocking even after a readiness notification and treats `EAGAIN` as an ordinary race outcome.

### **I/O Multiplexing and Readiness**

Creating one blocking thread per connection gives each flow a simple sequential program, but large numbers of mostly idle connections make thread stacks, scheduler activity, and shared-state synchronization expensive. **I/O multiplexing** lets one thread wait for state changes across many descriptors and then run the handlers associated with the ready subset.

A descriptor is **readable** when a read-like operation would not block. This includes more than "payload bytes exist": EOF, a pending error, or an incoming connection can also make a descriptor readable. A descriptor is **writable** when a write-like operation can accept at least some progress without blocking. It does not promise that an entire application message fits, that the peer will receive it, or that future writes remain nonblocking.

An event loop repeatedly performs five jobs:

1. wait for descriptor readiness, a timer deadline, or queued cross-thread work;
2. map each returned event to application state;
3. perform nonblocking operations until progress stops or a fairness budget is reached;
4. update which events are interesting; and
5. return to the wait point without retaining stale pointers or starving other work.

![A readiness loop waits, dispatches, drains nonblocking operations, updates state, and runs timers.](assets/readiness-event-loop-animated.svg){fig-alt="An animated token moves around a readiness event loop from waiting to ready-set dispatch, nonblocking draining, interest updates, timer work, and back to waiting." width="96%"}

*Figure: original explanatory animation informed by the [OSTEP event-based concurrency chapter](https://pages.cs.wisc.edu/~remzi/OSTEP/threads-events.pdf) and Linux [`epoll(7)`](https://man7.org/linux/man-pages/man7/epoll.7.html).*

The handler budget is not an implementation detail. If one busy connection is drained forever, timers and other connections never run. If a handler processes too little, the program pays more event-loop and system-call overhead. Mature loops therefore combine a byte/operation budget with explicit output-queue limits and timer deadlines.

#### **select, poll, epoll, and kqueue**

The major readiness APIs differ in where the **interest set** lives and how the ready subset is returned.

![select and poll pass interests on each wait, while epoll and kqueue maintain persistent kernel registrations.](assets/multiplexing-api-comparison.svg){fig-alt="Four columns compare select bit sets, poll arrays, Linux epoll persistent ready lists, and BSD kqueue filters by interest representation, kernel work, result delivery, and important constraints." width="97%"}

*Figure: original explanatory comparison based on Linux [`select(2)`](https://man7.org/linux/man-pages/man2/select.2.html), [`poll(2)`](https://man7.org/linux/man-pages/man2/poll.2.html), [`epoll(7)`](https://man7.org/linux/man-pages/man7/epoll.7.html), and FreeBSD [`kqueue(2)`](https://man.freebsd.org/cgi/man.cgi?kqueue%282%29).*

| API | Interest model | Returned work | Important property |
|---|---|---|---|
| `select()` | Caller supplies bit sets on every call; sets are modified on return | Application scans descriptors up to `nfds` | Widely portable but limited by `FD_SETSIZE` on common implementations; Linux recommends modern alternatives |
| `poll()` | Caller supplies an array of `pollfd` records on every call | Kernel and application inspect the array | No bit-set limit and simple semantics; repeated scan/copy cost grows with registered descriptors |
| `epoll` | Linux kernel stores a persistent interest set and ready list | `epoll_wait()` returns ready event records | Scales well for many mostly idle descriptors; supports level, edge, and one-shot modes |
| `kqueue` | BSD-family kernel stores filters registered through `kevent()` | `kevent()` returns triggered records and filter metadata | General event filters cover descriptors, signals, timers, processes, and more |

`select()` and `poll()` are still good choices for small, portable programs because their state is explicit in each call. `epoll` and `kqueue` avoid resubmitting a large mostly idle set and return records only for reported events, but they introduce persistent kernel registration that must be kept consistent with application object lifetime.

Readiness delivery has two common modes:

- **Level-triggered:** a condition is reported while it remains true. If data remains unread, the next wait reports readability again. This is easier to reason about but can repeatedly return a descriptor the handler only partially services.
- **Edge-triggered:** a transition produces notification. After receiving an edge, the handler must drain the nonblocking operation until `EAGAIN`; otherwise unread work may remain without another transition to wake the loop.

`EPOLLONESHOT` adds an explicit rearm step and can help coordinate one descriptor across worker threads, but the application must not forget to rearm it. Errors and hangups must be handled even when the originally requested data event is absent. In particular, a hangup can coexist with buffered bytes, so a handler should drain available input before treating EOF as final.

The following Linux example uses an edge-triggered `epoll` loop around a nonblocking pipe. A child writes three bursts. The parent drains until `EAGAIN`, recognizes EOF, and never assumes that one event corresponds to one complete message.

<details>
<summary><strong>C example: edge-triggered epoll with a nonblocking pipe</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <fcntl.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/epoll.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <time.h>
#include <unistd.h>

static void die(const char *what) {
    perror(what);
    exit(EXIT_FAILURE);
}

static void child_write_all(int fd, const char *text) {
    size_t used = 0;
    size_t length = strlen(text);

    while (used < length) {
        ssize_t n = write(fd, text + used, length - used);
        if (n > 0) {
            used += (size_t)n;          // A successful write may be partial.
        } else if (n < 0 && errno == EINTR) {
            continue;                   // Retry only because this demo has no cancellation policy.
        } else if (n < 0 && errno == EAGAIN) {
            struct timespec pause = {.tv_sec = 0, .tv_nsec = 1000000};
            nanosleep(&pause, NULL);     // Real code would wait for writable readiness.
        } else {
            die("write");
        }
    }
}

int main(void) {
    int pipefd[2];
    if (pipe2(pipefd, O_NONBLOCK | O_CLOEXEC) == -1) {
        die("pipe2");
    }

    pid_t child = fork();
    if (child == -1) die("fork");

    if (child == 0) {
        close(pipefd[0]);
        const char *bursts[] = {"first burst\n", "second burst\n", "final burst\n"};

        for (size_t i = 0; i < sizeof bursts / sizeof bursts[0]; ++i) {
            child_write_all(pipefd[1], bursts[i]);
            struct timespec pause = {.tv_sec = 0, .tv_nsec = 150000000};
            nanosleep(&pause, NULL);     // Make separate readiness transitions visible.
        }

        close(pipefd[1]);                // Reader will eventually observe EOF.
        _exit(EXIT_SUCCESS);
    }

    close(pipefd[1]);
    int epfd = epoll_create1(EPOLL_CLOEXEC);
    if (epfd == -1) die("epoll_create1");

    struct epoll_event registration = {
        .events = EPOLLIN | EPOLLET,
        .data.fd = pipefd[0],
    };
    if (epoll_ctl(epfd, EPOLL_CTL_ADD, pipefd[0], &registration) == -1) {
        die("epoll_ctl");
    }

    int finished = 0;
    while (!finished) {
        struct epoll_event events[8];
        int ready = epoll_wait(epfd, events, 8, -1);
        if (ready < 0 && errno == EINTR) continue;
        if (ready < 0) die("epoll_wait");

        for (int i = 0; i < ready; ++i) {
            if (events[i].data.fd != pipefd[0]) continue;

            // EPOLLET rule: drain every currently available byte.
            for (;;) {
                char buffer[16];
                ssize_t n = read(pipefd[0], buffer, sizeof buffer);

                if (n > 0) {
                    // write() to stdout is kept simple for the teaching example.
                    if (write(STDOUT_FILENO, buffer, (size_t)n) < 0) die("stdout");
                } else if (n == 0) {
                    finished = 1;        // All writers closed and buffered data is gone.
                    break;
                } else if (errno == EINTR) {
                    continue;
                } else if (errno == EAGAIN || errno == EWOULDBLOCK) {
                    break;               // Fully drained; return to epoll_wait().
                } else {
                    die("read");
                }
            }

            if (events[i].events & EPOLLERR) {
                fprintf(stderr, "epoll reported an error condition\n");
            }
            // EPOLLHUP is not handled before draining because bytes can coexist with HUP.
        }
    }

    close(pipefd[0]);
    close(epfd);
    if (waitpid(child, NULL, 0) == -1) die("waitpid");
    return EXIT_SUCCESS;
}
```

Compile and run on Linux:

```bash
cc -std=c11 -Wall -Wextra -O2 epoll_pipe.c -o epoll_pipe
./epoll_pipe
```

</details>

The example uses a pipe to isolate readiness semantics from network protocols. A production server additionally needs per-connection input framing, bounded output queues, idle timeouts, signal handling, safe descriptor removal, and a policy for distributing CPU-heavy work.

### **Asynchronous I/O and Completion Interfaces**

Nonblocking and asynchronous are related but not synonymous. **Nonblocking** describes what an individual operation does when it would have to wait. **Asynchronous I/O** describes a lifecycle in which the application submits an operation and later receives that operation's result.

![Readiness precedes an operation attempt, while completion follows a submitted operation.](assets/readiness-vs-completion.svg){fig-alt="Two timelines compare registering interest and receiving a readiness hint before calling read with submitting an asynchronous read and later receiving a correlated completion result." width="96%"}

*Figure: original explanatory diagram based on Linux [`epoll(7)`](https://man7.org/linux/man-pages/man7/epoll.7.html) and [`io_uring(7)`](https://man7.org/linux/man-pages/man7/io_uring.7.html).*

Three implementation strategies are common:

| Strategy | What application code sees | How waiting happens underneath | Main trade-off |
|---|---|---|---|
| Blocking calls | One call returns one result | Calling thread sleeps | Simple control flow; many blocked flows can require many threads |
| Readiness + nonblocking calls | Event says which object is worth trying | Kernel tracks object state; application performs operation | Efficient for many sockets/pipes; application owns a state machine |
| Completion API | Submit operation now, receive result later | Kernel, device, or worker completes operation | Natural batching and in-flight concurrency; buffer/request lifetime is harder |

A thread pool can expose a completion-style application API while workers execute blocking calls. This is often practical for regular-file I/O and libraries that are not natively asynchronous. It bounds the number of blocked kernel threads but does not eliminate thread scheduling, stack memory, or pool saturation. A true kernel completion interface can issue supported operations without dedicating one userspace worker per wait.

Completions also change correlation. In readiness code, a descriptor identifies an object and application state decides what to try next. In completion code, the application normally attaches an operation-specific identity because several reads, writes, timeouts, and cancellations can be in flight on one object and may complete out of order.

#### **io_uring and Submission-Completion Queues**

Linux `io_uring` exposes a **submission queue (SQ)** and **completion queue (CQ)** through memory shared between userspace and the kernel. Userspace prepares a submission queue entry (SQE), publishes its index to the SQ ring, and asks the kernel to consume submissions. The kernel later publishes a completion queue entry (CQE) containing a result, flags, and the application-provided `user_data` value.

![io_uring carries SQEs toward the kernel and correlated CQEs back to userspace.](assets/io-uring-sq-cq-animated.svg){fig-alt="An animated SQE moves from userspace preparation through the shared submission ring into kernel and device execution, then a CQE carrying result and user_data returns through the completion ring." width="96%"}

*Figure: original explanatory animation based on Linux [`io_uring(7)`](https://man7.org/linux/man-pages/man7/io_uring.7.html), [`io_uring_setup(2)`](https://man7.org/linux/man-pages/man2/io_uring_setup.2.html), and [`io_uring_enter(2)`](https://man7.org/linux/man-pages/man2/io_uring_enter.2.html).*

The shared rings reduce repeated copying of operation descriptions and can amortize system calls across batches. Optional modes can poll submissions, register frequently used files and buffers, link operations, attach timeouts, or produce multiple completions from a multishot request. These features are optimizations around the same ownership protocol:

1. allocate request state and choose a unique `user_data` identity;
2. keep every referenced buffer, file, and metadata object valid;
3. prepare and publish the SQE with correct memory ordering;
4. submit or wake the configured kernel consumer;
5. process CQEs, noting that order can differ from submission order;
6. interpret `cqe->res` as a nonnegative result or negative errno value; and
7. release/reuse resources only after the terminal completion or confirmed cancellation.

Shared command rings do **not** imply universal zero-copy data transfer. An operation may still copy between a user buffer and kernel cache, protocol buffer, or intermediate representation. Registered buffers can reduce mapping/pinning overhead for supported paths, while `splice`, direct I/O, device DMA, and protocol-specific facilities address different copies. Each claim must name which copy and which path were removed.

The following example submits one asynchronous file read with `liburing`. It deliberately stores a pointer to request state in `user_data` and keeps the buffer alive until the CQE has been consumed.

<details>
<summary><strong>C example: submit and reap one io_uring read</strong></summary>

```c
#include <errno.h>
#include <fcntl.h>
#include <liburing.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

struct request {
    char *buffer;
    size_t capacity;
};

static void fail_liburing(const char *what, int negative_errno) {
    fprintf(stderr, "%s: %s\n", what, strerror(-negative_errno));
    exit(EXIT_FAILURE);
}

int main(int argc, char **argv) {
    if (argc != 2) {
        fprintf(stderr, "usage: %s FILE\n", argv[0]);
        return EXIT_FAILURE;
    }

    int fd = open(argv[1], O_RDONLY | O_CLOEXEC);
    if (fd == -1) {
        perror("open");
        return EXIT_FAILURE;
    }

    struct io_uring ring;
    int rc = io_uring_queue_init(8, &ring, 0);
    if (rc < 0) fail_liburing("io_uring_queue_init", rc);

    struct request *req = calloc(1, sizeof *req);
    if (req == NULL) {
        perror("calloc request");
        return EXIT_FAILURE;
    }
    req->capacity = 4096;
    req->buffer = malloc(req->capacity);
    if (req->buffer == NULL) {
        perror("malloc buffer");
        return EXIT_FAILURE;
    }

    struct io_uring_sqe *sqe = io_uring_get_sqe(&ring);
    if (sqe == NULL) {
        fprintf(stderr, "submission queue is full\n");
        return EXIT_FAILURE;
    }

    // Describe a read at file offset zero. The operation has not completed here.
    io_uring_prep_read(sqe, fd, req->buffer, req->capacity, 0);
    io_uring_sqe_set_data(sqe, req);  // Copied into CQE user_data for correlation.

    rc = io_uring_submit(&ring);
    if (rc < 0) fail_liburing("io_uring_submit", rc);
    if (rc != 1) {
        fprintf(stderr, "expected one submitted SQE, got %d\n", rc);
        return EXIT_FAILURE;
    }

    struct io_uring_cqe *cqe = NULL;
    rc = io_uring_wait_cqe(&ring, &cqe);
    if (rc < 0) fail_liburing("io_uring_wait_cqe", rc);

    struct request *completed = io_uring_cqe_get_data(cqe);
    int result = cqe->res;             // Nonnegative byte count or negative errno.

    if (result < 0) {
        fprintf(stderr, "asynchronous read: %s\n", strerror(-result));
    } else {
        size_t written = 0;
        while (written < (size_t)result) {
            ssize_t n = write(STDOUT_FILENO,
                              completed->buffer + written,
                              (size_t)result - written);
            if (n > 0) written += (size_t)n;
            else if (n < 0 && errno == EINTR) continue;
            else { perror("write stdout"); break; }
        }
    }

    io_uring_cqe_seen(&ring, cqe);     // Return this CQ slot to the kernel/user ring.
    free(completed->buffer);           // Safe only after the operation completed.
    free(completed);
    io_uring_queue_exit(&ring);
    close(fd);
    return result < 0 ? EXIT_FAILURE : EXIT_SUCCESS;
}
```

Compile on Linux after installing the `liburing` development package:

```bash
cc -std=c11 -Wall -Wextra -O2 uring_read.c -luring -o uring_read
./uring_read example.txt
```

</details>

Real `io_uring` programs must also handle queue saturation, short operations, CQ capacity, feature detection, cancellation races, and shutdown. A multishot SQE can produce several CQEs; conversely, flags that suppress successful completions can intentionally produce none. The request's documented terminal rule, not an assumption of "one SQE equals one CQE," must govern cleanup.

### **Buffering, Caching, and Backpressure**

A **buffer** temporarily holds data to reconcile differences in timing, transfer size, alignment, or execution context. A **cache** retains data because a future request may reuse it. The same memory can serve both roles: the page cache buffers writes and caches file contents, while a socket send buffer queues bytes for transmission but is not normally an application-visible content cache.

Buffering enables:

- a producer and consumer to run at different instants;
- small operations to be batched into efficient device transfers;
- interrupt handlers to hand off work quickly;
- double buffering, where one region is filled while another is consumed; and
- queueing that exposes device or pipeline parallelism.

It does not make sustained overload disappear. If arrivals average $\lambda$ bytes/s and the consumer sustainably serves only $\mu < \lambda$, queue occupancy grows until memory is exhausted unless admission is reduced or work is dropped. A bounded queue turns that hidden catastrophe into an explicit policy decision.

![A bounded queue uses high- and low-water marks to propagate pressure to a faster producer.](assets/buffering-backpressure.svg){fig-alt="A producer fills a bounded queue faster than a consumer drains it; at a high-water mark the system throttles admission, and a queue-occupancy graph shows the buffer filling then draining." width="96%"}

*Figure: original explanatory diagram applying Little's law and bounded-queue flow control to operating-system I/O.*

For a stable system, Little's law again gives

$$
L = \lambda W,
$$

where $L$ is average queued/in-flight work, $\lambda$ is average completed throughput, and $W$ is average time in the system. If throughput is fixed, doubling average queued work roughly doubles average residence time. A larger buffer can absorb a longer burst and reduce drops, but it can also hide congestion and produce **bufferbloat**, where requests wait far longer without improving service rate.

Backpressure mechanisms differ by interface:

| Layer | Pressure signal | Expected response |
|---|---|---|
| Nonblocking stream write | Partial count or `EAGAIN` | Retain only a bounded remainder; enable writable interest; stop reading upstream if needed |
| Pipe or socket producer | Blocking write sleeps | Scheduler prevents busy waiting, but application still needs cancellation and memory limits |
| Device submission ring | No free descriptor / queue full | Stop admission or select another queue; reap completions |
| Event-loop output queue | High-water mark | Disable producer input, shed work, or apply protocol flow control |
| Network stack | Congestion/loss/receiver window | Transport and application reduce sending rate |

High- and low-water marks prevent rapid on/off oscillation. Admission stops at the high mark and resumes only after occupancy falls below a lower mark. The gap provides **hysteresis**. A complete design also decides who is allowed to wait, how long, what can be discarded, and how pressure crosses process or protocol boundaries.

### **Character, Block, and Network Devices**

Device classes are abstractions chosen because different hardware benefits from different request shapes.

- A **character-style device** exposes a sequential byte stream or device-defined records. Terminals, serial links, and many sensors fit this model. `read()`, `write()`, and device-specific `ioctl()` commands dominate.
- A **block device** exposes fixed-size addressable storage units and supports requests at chosen offsets. The block layer can merge, prioritize, and dispatch multiple requests while a controller executes them concurrently.
- A **network device** exchanges packets with the kernel network stack through descriptor rings. Applications normally use sockets, which represent transport endpoints and protocol state rather than direct handles to a NIC.

![Character, block, and network paths differ in transfer unit, internal queues, and application abstraction.](assets/device-class-paths.svg){fig-alt="Three columns trace character devices from byte-oriented calls, block devices through request queues to addressable sectors, and network sockets through protocol queues to NIC descriptor rings." width="96%"}

*Figure: original explanatory diagram informed by Linux [device-number assignments](https://docs.kernel.org/admin-guide/devices.html), [block-layer statistics](https://docs.kernel.org/block/stat.html), and [NAPI](https://docs.kernel.org/networking/napi.html).*

| Property | Character-style | Block | Network |
|---|---|---|---|
| Natural transfer unit | Byte or device-specific record | Sector/block range | Packet/frame below the socket layer |
| Addressing | Usually sequential | Random offset/range | Flow and protocol addresses, not storage offsets |
| Important queues | Input/output buffers | Software request queues and hardware submission/completion queues | Socket buffers, qdisc, protocol queues, NIC rings |
| Reordering | Often preserves stream order | Scheduler/controller may reorder independent requests | Packets may reorder/drop; transport may reconstruct an ordered stream |
| Common application API | `read`, `write`, `ioctl` | Usually file-system or direct-I/O APIs | `socket`, `send`, `recv`, readiness/completion APIs |
| Dominant policy concerns | Latency, flow control, device state | Throughput, tail latency, fairness, durability boundary | Bursts, drops, CPU affinity, congestion, protocol state |

These classes are not physical laws. A single PCIe device can expose several functions and queue types. A virtual block device may ultimately send network requests. A network interface can support control channels that look character-like. The class determines which kernel subsystem owns policy and presents a stable contract; the driver still translates that contract to hardware.

### **Errors, Timeouts, Cancellation, and Hotplug**

An I/O error is rarely just "the function returned -1." The request may be queued, executing, partially transferred, completed but not yet observed, or affected by a reset. Recovery begins by defining the request state and who owns each resource in that state.

A **timeout** means a deadline passed before the observer received a terminal result. It does not prove the device performed no work. A write could have reached hardware while its completion interrupt was lost. A network request could have reached a remote service while its response was delayed. Retrying blindly can duplicate a non-idempotent side effect.

A **cancellation request** is also a race. If cancellation removes queued work before execution, a canceled result is appropriate. If completion wins, the original result may remain authoritative. Some devices can abort active commands; others require a queue or device reset that also affects unrelated requests. Correct APIs document whether cancellation is best-effort and how its own completion relates to the target operation.

![Timeout, cancellation, reset, completion, and hot unplug converge on one terminal ownership rule.](assets/io-error-cancel-hotplug.svg){fig-alt="A request state machine moves from prepared to submitted, in flight, completed, and reaped; timeout or cancellation can race with completion or require reset, while hot unplug drains every request exactly once." width="96%"}

*Figure: original explanatory diagram based on Linux USB [synchronous and asynchronous I/O](https://docs.kernel.org/driver-api/usb/usb.html), the [libATA error-handling model](https://docs.kernel.org/driver-api/libata.html), and [`io_uring(7)`](https://man7.org/linux/man-pages/man7/io_uring.7.html).*

The central invariant is:

> **Each submitted request reaches one terminal ownership transition, and its buffers, identifiers, and device references remain valid until that transition is observed.**

This invariant prevents double completion, use-after-free, stale CQEs, and a late interrupt operating on a replacement device. Generation numbers or unique request IDs help distinguish an old completion from new work that reused the same queue slot.

Hotplug applies the same reasoning to every outstanding request:

1. mark the device unavailable and stop accepting new work;
2. prevent new callbacks or interrupts from racing with teardown;
3. cancel, drain, or fail each outstanding request according to hardware capability;
4. wait for references and deferred handlers to leave critical sections;
5. release DMA mappings, queues, IRQs, MMIO ranges, and subsystem registration; and
6. report a stable error to applications still holding handles.

Errors should be classified before choosing a response:

| Error class | Example | Sensible first response |
|---|---|---|
| Transient resource pressure | Queue full, `EAGAIN`, temporary memory shortage | Apply backpressure or bounded retry |
| Request-specific input error | Invalid command, bad address, unsupported operation | Fail the request; do not reset healthy hardware |
| Media/link error | Unreadable block, link loss, CRC failure | Report detail; retry only under subsystem policy |
| Lost progress / timeout | No completion before deadline | Inspect queue/controller; abort or reset with race handling |
| Device removal | USB unplug, PCIe removal | Stop admission, fail/drain requests, detach safely |
| Driver invariant violation | Unknown completion ID, impossible queue index | Contain damage, record diagnostics, avoid memory corruption |

Retry policies need bounded attempts, deadlines, jitter where many clients can synchronize, and an idempotency argument. "Retry three times" is not a recovery design unless it explains what state may already have changed.

### **Observing I/O Behavior**

I/O latency is layered, so diagnosis should be layered too. Begin with the application's symptom and descend only when evidence points lower. A throughput graph without queue depth cannot distinguish efficient parallelism from growing delay. A device-utilization graph without application latency cannot show whether users are meeting their goals.

![Linux tools expose application, syscall, process, queue, interrupt, block, and network layers.](assets/linux-io-observability.svg){fig-alt="A layered Linux observability map connects application latency to strace, process I/O counters, scheduler statistics, interrupt and softirq counters, kernel tracing, block statistics, and network-device counters." width="96%"}

*Figure: original explanatory diagram based on Linux [`/proc` documentation](https://docs.kernel.org/filesystems/proc.html), [block statistics](https://docs.kernel.org/block/stat.html), and [kernel tracepoints](https://docs.kernel.org/core-api/tracepoint.html).*

Useful questions and evidence include:

| Question | Evidence | Important caution |
|---|---|---|
| Which call is waiting or failing? | `strace -ttT`, `perf trace`, application spans | A syscall duration can include sleeping and rescheduling |
| Is the process issuing the expected volume? | `/proc/$pid/io`, `pidstat -d` | `rchar/wchar` count syscall bytes; `read_bytes/write_bytes` describe storage-backed accounting and can differ |
| Are interrupts or deferred handlers concentrated? | `/proc/interrupts`, `/proc/softirqs`, CPU affinity | High counts are not automatically bad; compare useful work and latency |
| Is block queueing growing? | `iostat -xz`, `/sys/block/<dev>/stat`, block tracepoints | Utilization alone is not comparable across all parallel devices |
| Is the network dropping or erroring? | `ip -s link`, `/sys/class/net/*/statistics`, `ethtool -S` | Driver-specific counters require hardware documentation |
| Where is tail latency introduced? | ftrace/eBPF tracepoints with request IDs and timestamps | Tracing has overhead and often requires privileges |

<details>
<summary><strong>Shell workflow: inspect an I/O-bound process from the API boundary downward</strong></summary>

```bash
# 1. Observe relevant system calls and their elapsed time.
# Attach briefly in production; tracing every call can perturb the workload.
sudo strace -ttT -p "$PID" \
  -e trace=read,write,pread64,pwrite64,poll,ppoll,epoll_wait,io_uring_enter

# 2. Compare process-level logical I/O with storage-backed accounting.
cat "/proc/$PID/io"
pidstat -d -w -p "$PID" 1

# 3. Check whether interrupt or deferred-network work is concentrated by CPU.
cat /proc/interrupts
cat /proc/softirqs

# 4. Inspect block-device queueing and latency. Replace DEVICE explicitly.
iostat -xz 1
cat "/sys/block/$DEVICE/stat"

# 5. Inspect network-interface drops and driver counters. Replace IFACE explicitly.
ip -s link show dev "$IFACE"
sudo ethtool -S "$IFACE"

# 6. Use kernel tracing only after choosing a concrete question and time window.
# Available event names vary by kernel; list before enabling.
sudo perf list 'block:*'
sudo perf trace -p "$PID"
```

</details>

Interpret counters as transitions, not decorations. `/sys/block/<device>/stat` includes requests in flight and cumulative time fields; differences over a known interval are meaningful, whereas a single cumulative number is not a current latency. `/proc/interrupts` can reveal queue affinity, but a balanced count is not necessarily optimal if it destroys cache locality. Correlating one request identity across layers is usually more informative than collecting every available metric.

### **Following the Pipeline Through the I/O Stack**

The following representative path is a blocking, buffered regular-file `read()` whose requested page is absent from the page cache. It intentionally leaves file-system offset-to-block mapping as a black box for the next chapter:

1. The application calls `read(fd, buffer, count)` and enters the kernel.
2. The kernel resolves `fd`, checks permissions and the userspace destination, and asks the file object for bytes at the current offset.
3. The page cache does not contain the needed page, so the file and block layers construct lower-level work.
4. The driver allocates request metadata, maps destination pages for DMA, fills a submission descriptor, orders memory, and rings a doorbell.
5. The calling task sleeps on a wait queue. The CPU runs other work while the controller fetches the command.
6. The storage device services the request and DMA writes data into mapped memory.
7. The controller publishes a completion and signals an interrupt, or a host poll loop observes it.
8. The driver and block layer validate the completion, unmap DMA state, mark the page valid, and wake the waiter.
9. After the scheduler runs the task, the kernel copies requested bytes to userspace, advances the file position as specified, and returns a byte count.

![A cache-missing file read crosses the syscall, cache, block, driver, device, completion, and scheduler layers.](assets/io-request-end-to-end-timeline.svg){fig-alt="A swimlane timeline follows a buffered read from userspace through descriptor lookup, page-cache miss, driver and DMA submission, device service, interrupt completion, page validation, wakeup, copy, and return." width="98%"}

*Figure: original explanatory timeline synthesizing the Linux [VFS](https://docs.kernel.org/filesystems/vfs.html), [DMA API](https://docs.kernel.org/core-api/dma-api-howto.html), [generic IRQ](https://docs.kernel.org/core-api/genericirq.html), and [block statistics](https://docs.kernel.org/block/stat.html) models.*

One approximate latency decomposition is

$$
T_{read} = T_{entry} + T_{lookup} + T_{queue} + T_{device}
         + T_{completion} + T_{copy} + T_{schedule}.
$$

- $T_{entry}$ covers privilege transition and basic system-call handling.
- $T_{lookup}$ covers descriptor/object work and locating the requested cached/backing state.
- $T_{queue}$ covers software and hardware waiting before service begins.
- $T_{device}$ is controller/device service and payload transfer.
- $T_{completion}$ covers IRQ/poll detection and subsystem completion work.
- $T_{copy}$ moves requested bytes between kernel-managed storage and the user buffer on this buffered path.
- $T_{schedule}$ is delay until a woken task actually runs.

The sum explains one request but not aggregate throughput, because stages overlap across requests. While one request is at the device, a CPU can prepare another and complete a third. Queue depth enables overlap until a bottleneck saturates; beyond that point, additional depth primarily raises $T_{queue}$.

Nearby variants alter the route:

| Variant | What changes |
|---|---|
| Page-cache hit | The path ends before block submission and physical device service |
| `mmap()` access | A page fault initiates population; the final user access is a restarted load/store rather than a `read()` copy |
| Direct I/O | The path attempts to transfer between device and suitably aligned user pages while bypassing ordinary page-cache data flow; alignment and coherence rules tighten |
| Readiness loop | Appropriate for sockets/pipes; it waits for object state and then issues nonblocking operations |
| `io_uring` | Submission and completion are queue records; the calling thread need not block and several operations can remain in flight |
| Worker pool | A worker follows a blocking path while another thread receives a future/promise-style completion |

The most useful debugging habit is to name the variant before interpreting measurements. "I/O is slow" is not yet a hypothesis; "block requests spend 12 ms in the software queue while device service remains 0.4 ms" is one.

### **Comparison and Summary**

The I/O stack separates a stable application contract from device-specific execution, but every abstraction preserves several facts: operations can be partial, devices progress concurrently, results arrive later, queues are finite, and failure can race with completion.

| Decision | Prefer this when | Cost or obligation |
|---|---|---|
| Blocking call | Flow is sequential, concurrency is modest, or a worker pool bounds waits | One thread context is occupied by each active blocking operation |
| Nonblocking + readiness | Many descriptors are usually idle and operations are naturally readiness-driven | Application must maintain state, drain correctly, and implement backpressure |
| Completion API | Many operations should be batched/in flight, especially supported file/network operations | Buffer lifetime, request correlation, cancellation, and queue management are explicit |
| Interrupt notification | Events are sparse or a queue has gone idle | Fixed interrupt and scheduling cost; coalescing adds latency |
| Polling | Event rate is sustained and a CPU budget/core is available | Burns CPU while checking and must preserve fairness |
| Hybrid interrupt/poll | Workload alternates between idle and bursts | More policy and tuning; requires correct transition between modes |
| PIO | Payload is tiny or hardware is extremely simple | CPU moves every word |
| DMA | Payload is large or frequent | Mapping, synchronization, isolation, and ownership become critical |

Common misconceptions can now be corrected precisely:

| Misconception | More accurate statement |
|---|---|
| "A file descriptor is a file." | It is a process-local handle to a kernel object, which may be a file, pipe, socket, event source, or device. |
| "Blocking I/O blocks the CPU." | It can suspend the calling thread; the scheduler normally runs other work. |
| "Readable means bytes are available." | It means a read-like operation should not block; EOF and errors can also satisfy the condition. |
| "One readiness event means one message." | Readiness reports state, not application framing. Drain according to the object's contract. |
| "Interrupts are always faster than polling." | Interrupts save idle CPU; polling can reduce per-event overhead at sustained high rates. |
| "DMA means zero CPU involvement." | The CPU still prepares mappings/descriptors, handles completion, and may copy at other layers. |
| "io_uring is zero-copy." | Its shared SQ/CQ reduces command/completion overhead; payload copies depend on the selected operation and data path. |
| "A timeout means nothing happened." | It means no terminal result was observed by the deadline; the operation may have partially or fully executed. |
| "A bigger buffer fixes overload." | It postpones pressure and increases possible waiting; sustained arrival must not exceed sustainable service. |

A sound I/O design can answer these questions:

1. What operation is submitted, and which object owns its semantics?
2. Which agent moves the payload, and through which address/mapping?
3. How is completion detected and correlated with the request?
4. When may each descriptor, buffer, and device reference be reused?
5. How are partial progress, EOF, errors, deadlines, cancellation, and hotplug represented?
6. Where is queue capacity bounded, and how does backpressure reach the producer?
7. Which measurements separate syscall, queue, device, completion, and scheduling time?

With those answers, event-driven execution becomes less mysterious. It is the explicit organization of work that was always asynchronous at the hardware boundary: submit, relinquish ownership, let independent agents progress, observe one terminal result, and propagate pressure or failure without losing track of state.
